In [ ]:
import numpy as np
import pandas as pd

# =========================================================================
# Configuration & Setup
# =========================================================================
valid_sub_ids = [0, 3, 4, 5, 6]  #
accuracy_categories = ["Hit", "Miss", "False Alarm", "Correct Rejection"]  #
bin_labels = {  #[cite: 1]
    1: "0-10%",
    2: "15-25%",
    3: "30-40%",
    4: "45-55%",
    5: "60-70%",
    6: "75-100%",
}

# Face-selective electrode channel mapping per subject
face_channels_per_sub = {
    0: [0, 25, 30, 35, 39],  # Subject 1
    3: [23, 24],  # Subject 4
    4: [15],  # Subject 5
    5: [6, 9, 15, 16, 17, 23, 24, 25, 30],  # Subject 6
    6: [8, 9, 36, 38, 39],  # Subject 7
}

# Epoch timing parameters (1000 Hz sampling rate -> 1 sample = 1 ms)
n_pre = 200  # 200 ms pre-stimulus baseline[cite: 1]
n_post = 800  # 800 ms post-stimulus window[cite: 1]
n_samples = n_pre + n_post  # 1000 total time points[cite: 1]
time_vec = np.arange(-n_pre, n_post)  # Time vector (-200 ms to +799 ms)[cite: 1]

# =========================================================================
# Master Processing Loop per Subject
# =========================================================================
group_face_erps = {}

for sub_id in valid_sub_ids:
    dat2 = alldat[sub_id][1]  #[cite: 1]
    face_chans = face_channels_per_sub[sub_id]

    # ---------------------------------------------------------------------
    # Step 1: Noise Level Binning (1 to 6)
    # ---------------------------------------------------------------------
    stim_noise = np.squeeze(dat2["stim_noise"])  #[cite: 1]
    conditions = [  #[cite: 1]
        (stim_noise >= 0) & (stim_noise <= 10),
        (stim_noise >= 15) & (stim_noise <= 25),
        (stim_noise >= 30) & (stim_noise <= 40),
        (stim_noise >= 45) & (stim_noise <= 55),
        (stim_noise >= 60) & (stim_noise <= 70),
        (stim_noise >= 75) & (stim_noise <= 100),
    ]
    choices = [1, 2, 3, 4, 5, 6]  #[cite: 1]
    noise_bins = np.select(conditions, choices, default=0)  #[cite: 1]
    dat2["noise_bin"] = noise_bins  #[cite: 1]

    # ---------------------------------------------------------------------
    # Step 2: Keypress Detection & Behavioral Outcome Classification
    # ---------------------------------------------------------------------
    t_on = np.squeeze(dat2["t_on"]).astype(int)  #[cite: 1]
    t_off = np.squeeze(dat2["t_off"])  #[cite: 1]
    stim_cat = np.squeeze(dat2["stim_cat"])  # 1 = House, 2 = Face[cite: 1]
    key_presses = np.squeeze(dat2["key_press"])  #[cite: 1]
    n_trials = len(t_on)  #[cite: 1]

    has_keypress = np.zeros(n_trials, dtype=bool)  #[cite: 1]
    for i in range(n_trials):
        start_time = t_on[i]  #[cite: 1]
        end_time = t_on[i + 1] if i < n_trials - 1 else t_off[i]  #[cite: 1]
        if key_presses.size > 0:
            has_keypress[i] = np.any(  #[cite: 1]
                (key_presses >= start_time) & (key_presses < end_time)
            )

    accuracy_label = np.empty(n_trials, dtype=object)  #[cite: 1]
    is_face = stim_cat == 2  #[cite: 1]
    is_house = stim_cat == 1  #[cite: 1]

    accuracy_label[is_face & has_keypress] = "Hit"  #[cite: 1]
    accuracy_label[is_face & ~has_keypress] = "Miss"  #[cite: 1]
    accuracy_label[is_house & has_keypress] = "False Alarm"  #[cite: 1]
    accuracy_label[is_house & ~has_keypress] = "Correct Rejection"  #[cite: 1]

    dat2["has_keypress"] = has_keypress  #[cite: 1]
    dat2["accuracy_label"] = accuracy_label  #[cite: 1]

    # ---------------------------------------------------------------------
    # Step 3: Epoching & Baseline Correction
    # ---------------------------------------------------------------------
    V = dat2["V"]  # Continuous voltage matrix (time, channels)[cite: 1]
    n_channels = V.shape[1]  #[cite: 1]
    epochs = np.zeros(
        (n_trials, n_samples, n_channels), dtype=np.float32
    )  #[cite: 1]

    for i in range(n_trials):
        onset = t_on[i]  #[cite: 1]
        raw_epoch = V[
            onset - n_pre : onset + n_post, :
        ]  # 1000 ms window[cite: 1]
        baseline = np.mean(
            raw_epoch[:n_pre, :], axis=0, keepdims=True
        )  # -200ms to 0ms mean[cite: 1]
        epochs[i] = raw_epoch - baseline  # Subtract baseline[cite: 1]

    dat2["epochs"] = epochs  #[cite: 1]
    dat2["time_vec"] = time_vec  #[cite: 1]

    # ---------------------------------------------------------------------
    # Step 4: Calculate Face-Selective ERPs (Bins x Accuracy Outcomes)
    # ---------------------------------------------------------------------
    erp_by_condition = {}
    subj_face_erp = {}

    for b in range(1, 7):
        erp_by_condition[b] = {}
        subj_face_erp[b] = {}

        for cat in accuracy_categories:
            cond_mask = (noise_bins == b) & (accuracy_label == cat)  #[cite: 1]
            n_cond_trials = np.sum(cond_mask)

            if n_cond_trials > 0:
                # Epoch average across matching trials[cite: 1]
                avg_epoch = np.mean(epochs[cond_mask], axis=0)  # (1000, chans)
                erp_by_condition[b][cat] = avg_epoch  #[cite: 1]

                # Filter exclusively for face-selective channels and take spatial average
                subj_face_erp[b][cat] = np.nanmean(
                    avg_epoch[:, face_chans], axis=1
                )
            else:
                erp_by_condition[b][cat] = np.full(  #[cite: 1]
                    (n_samples, n_channels), np.nan
                )
                subj_face_erp[b][cat] = np.full((n_samples,), np.nan)

    dat2["erp_by_condition"] = erp_by_condition  #[cite: 1]
    dat2["subj_face_erp"] = subj_face_erp
    group_face_erps[sub_id] = subj_face_erp

    # ---------------------------------------------------------------------
    # Step 5: Overall Subject Grand Mean ERP (Face Channels Only)
    # ---------------------------------------------------------------------
    # Averaged across ALL trials (axis 0) and face-selective channels (axis 2)[cite: 1]
    subject_grand_mean_face = np.nanmean(epochs[:, :, face_chans], axis=(0, 2))
    dat2["subject_grand_mean_face"] = subject_grand_mean_face

    print(
        f"Subject {sub_id} complete | Face Channels: {face_chans} | Grand Mean Shape: {subject_grand_mean_face.shape}"
    )

# =========================================================================
# Step 6: Compute Group Grand Mean ERP across Subjects (Face Channels)
# =========================================================================
group_grand_mean_face = {}

for b in range(1, 7):
    group_grand_mean_face[b] = {}
    for cat in accuracy_categories:
        # Stack face ERPs across all 5 valid subjects -> Shape: (5 subjects, 1000 ms)
        subj_signals = [
            group_face_erps[sub_id][b][cat] for sub_id in valid_sub_ids
        ]
        # Group average across subjects (axis 0)
        group_grand_mean_face[b][cat] = np.nanmean(
            np.array(subj_signals), axis=0
        )

print(
    "\nProcessing Complete. Data stored in `alldat[sub_id][1]` and `group_grand_mean_face`."
)

In [ ]:
import matplotlib.cm as cm
import matplotlib.pyplot as plt

# Colormap for the 6 noise bins (purple = low noise, yellow = high noise)
colors = cm.viridis(np.linspace(0, 0.9, 6))

# =========================================================================
# Figure 1: Group Grand Mean ERPs for Face Channels (Hits vs Misses)
# =========================================================================
fig, (ax_hit, ax_miss) = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for b in range(1, 7):
    color = colors[b - 1]
    label = f"Bin {b} ({bin_labels[b]})"  #[cite: 1]

    hit_signal = group_grand_mean_face[b]["Hit"]
    miss_signal = group_grand_mean_face[b]["Miss"]

    if not np.all(np.isnan(hit_signal)):
        ax_hit.plot(
            time_vec, hit_signal, label=label, color=color, linewidth=2
        )

    if not np.all(np.isnan(miss_signal)):
        ax_miss.plot(
            time_vec, miss_signal, label=label, color=color, linewidth=2
        )

# Format Hits Panel
ax_hit.axvline(0, color="k", linestyle="--", alpha=0.7, label="Stimulus Onset")
ax_hit.axhline(0, color="gray", linestyle=":", alpha=0.5)
ax_hit.set_title(
    "Group Grand Mean — Hits (Detected Faces)", fontsize=12, fontweight="bold"
)
ax_hit.set_xlabel("Time (ms)")
ax_hit.set_ylabel("Voltage (µV)")
ax_hit.grid(True, linestyle="--", alpha=0.3)
ax_hit.legend(loc="upper left", title="Noise Bins", frameon=True)

# Format Misses Panel
ax_miss.axvline(0, color="k", linestyle="--", alpha=0.7)
ax_miss.axhline(0, color="gray", linestyle=":", alpha=0.5)
ax_miss.set_title(
    "Group Grand Mean — Misses (Unseen Faces)", fontsize=12, fontweight="bold"
)
ax_miss.set_xlabel("Time (ms)")
ax_miss.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

# =========================================================================
# Figure 2: Subject-by-Subject Grid (5 Subjects x 4 Behavioral Outcomes)
# =========================================================================
fig, axes = plt.subplots(
    len(valid_sub_ids), 4, figsize=(20, 14), sharex=True, sharey="row"  #[cite: 1]
)

for row_idx, sub_id in enumerate(valid_sub_ids):  #[cite: 1]
    face_chans = face_channels_per_sub[sub_id]

    for col_idx, cat in enumerate(accuracy_categories):  #[cite: 1]
        ax = axes[row_idx, col_idx]

        for b in range(1, 7):
            signal = group_face_erps[sub_id][b][cat]
            color = colors[b - 1]
            label = f"Bin {b} ({bin_labels[b]})"  #[cite: 1]

            if not np.all(np.isnan(signal)):
                ax.plot(
                    time_vec, signal, label=label, color=color, linewidth=1.5
                )

        ax.axvline(0, color="k", linestyle="--", alpha=0.6)
        ax.axhline(0, color="gray", linestyle=":", alpha=0.5)
        ax.grid(True, linestyle="--", alpha=0.3)

        # Row 1 Titles
        if row_idx == 0:
            ax.set_title(cat, fontsize=12, fontweight="bold")

        # Y-Axis Label per Subject
        if col_idx == 0:
            ax.set_ylabel(
                f"Subject {sub_id}\nCh: {face_chans}\nVoltage (µV)",
                fontsize=10,
            )

        # Legend on top-left plot only
        if row_idx == 0 and col_idx == 0:
            ax.legend(
                loc="upper left", title="Noise Level", fontsize="x-small"
            )

for col_idx in range(4):
    axes[-1, col_idx].set_xlabel("Time (ms)")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np

# =========================================================================
# Step 1: Calculate Face-Selective ERPs per Noise Bin (Collapsing Outcomes)
# =========================================================================
valid_sub_ids = [0, 3, 4, 5, 6]  #[cite: 1]
bin_labels = {  #[cite: 1]
    1: "0-10%",
    2: "15-25%",
    3: "30-40%",
    4: "45-55%",
    5: "60-70%",
    6: "75-100%",
}

face_channels_per_sub = {
    0: [0, 25, 30, 35, 39],  # Subject 1
    3: [23, 24],  # Subject 4
    4: [15],  # Subject 5
    5: [6, 9, 15, 16, 17, 23, 24, 25, 30],  # Subject 6
    6: [8, 9, 36, 38, 39],  # Subject 7
}

colors = cm.viridis(np.linspace(0, 0.9, 6))
time_vec = alldat[valid_sub_ids[0]][1]["time_vec"]  #[cite: 1]

subject_noise_erps = {}

for sub_id in valid_sub_ids:
    dat2 = alldat[sub_id][1]  #[cite: 1]
    epochs = dat2["epochs"]  # Shape: (n_trials, 1000, n_channels)[cite: 1]
    noise_bins = np.squeeze(dat2["noise_bin"])  # Shape: (n_trials,)[cite: 1]
    face_chans = face_channels_per_sub[sub_id]

    subj_erps = {}

    for b in range(1, 7):
        bin_mask = noise_bins == b
        n_trials_bin = np.sum(bin_mask)

        if n_trials_bin > 0:
            # Average across all trials in this bin regardless of outcome[cite: 1]
            avg_epoch = np.mean(epochs[bin_mask], axis=0)
            # Spatial average across face-selective channels
            subj_erps[b] = np.nanmean(avg_epoch[:, face_chans], axis=1)
        else:
            subj_erps[b] = np.full((epochs.shape[1],), np.nan)

    dat2["face_erp_by_noise"] = subj_erps
    subject_noise_erps[sub_id] = subj_erps

# =========================================================================
# Step 2: Calculate Group Grand Mean across Subjects per Noise Bin
# =========================================================================
group_grand_mean_noise = {}

for b in range(1, 7):
    subj_signals = [subject_noise_erps[sub_id][b] for sub_id in valid_sub_ids]
    group_grand_mean_noise[b] = np.nanmean(np.array(subj_signals), axis=0)

# =========================================================================
# Step 3: Single Layout Plotting (1 Top Grand Mean + 5 Subject Subplots)
# =========================================================================
fig = plt.figure(figsize=(20, 9))
gs = gridspec.GridSpec(2, 5, figure=fig, height_ratios=[1.3, 1])

# --- Top Plot: Group Grand Mean (Spans all 5 columns) ---
ax_group = fig.add_subplot(gs[0, :])

for b in range(1, 7):
    signal = group_grand_mean_noise[b]
    color = colors[b - 1]
    label = f"Bin {b} ({bin_labels[b]})"  #[cite: 1]

    if not np.all(np.isnan(signal)):
        ax_group.plot(
            time_vec, signal, label=label, color=color, linewidth=2.2
        )

ax_group.axvline(
    0, color="k", linestyle="--", alpha=0.7, label="Stimulus Onset"
)
ax_group.axhline(0, color="gray", linestyle=":", alpha=0.5)
ax_group.set_title(
    "Group Grand Mean — Face Selective Channels (All Trial Outcomes Combined)",
    fontsize=13,
    fontweight="bold",
)
ax_group.set_ylabel("Voltage (µV)")
ax_group.grid(True, linestyle="--", alpha=0.3)
ax_group.legend(loc="upper left", title="Noise Bins", ncol=3, frameon=True)

# --- Bottom Plots: Individual Subjects (1 column per subject) ---
for idx, sub_id in enumerate(valid_sub_ids):
    ax = fig.add_subplot(gs[1, idx], sharex=ax_group)
    face_chans = face_channels_per_sub[sub_id]

    for b in range(1, 7):
        signal = subject_noise_erps[sub_id][b]
        color = colors[b - 1]
        label = f"Bin {b} ({bin_labels[b]})"  #[cite: 1]

        if not np.all(np.isnan(signal)):
            ax.plot(time_vec, signal, label=label, color=color, linewidth=1.8)

    ax.axvline(0, color="k", linestyle="--", alpha=0.6)
    ax.axhline(0, color="gray", linestyle=":", alpha=0.5)
    ax.set_title(
        f"Subject {sub_id + 1}\nCh: {face_chans}",
        fontsize=11,
        fontweight="bold",
    )
    ax.set_xlabel("Time (ms)")
    ax.grid(True, linestyle="--", alpha=0.3)

    if idx == 0:
        ax.set_ylabel("Voltage (µV)")

plt.tight_layout()
plt.show()